# 8.2 STRIPS planning and sub goals: practice notebook

Describe actions by what they need, add and delete, and search for plans forwards and backwards. Build planning heuristics by relaxing the problem, and see with the Sussman anomaly why solving sub goals one at a time can fail.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Forward search on the Sussman anomaly

In [ ]:
# Forward (progression) search in the blocks world. A state maps each block to what it sits on.
def moves(state):
    below = set(state.values())                 # blocks with something on top are not clear
    free = [b for b in sorted(state) if b not in below]
    out = []
    for b in free:
        if state[b] != "Table":
            out.append(("MoveToTable(" + b + ", " + state[b] + ")", b, "Table"))
        for y in free:
            if y != b:
                out.append(("Move(" + b + ", " + state[b] + ", " + y + ")", b, y))
    return out

start = {"A": "Table", "B": "Table", "C": "A"}      # the Sussman anomaly
queue = [(start, [])]
seen = [start]
while queue:
    state, plan = queue.pop(0)
    if state["A"] == "B" and state["B"] == "C":   # goal: On(A, B) and On(B, C)
        print("plan:", plan)
        print("states seen:", len(seen))
        break
    for name, b, to in moves(state):
        new = dict(state)
        new[b] = to
        if new not in seen:
            seen.append(new)
            queue.append((new, plan + [name]))

### Regressing a goal through actions

In [ ]:
# Backward (regression) search, one step: which goal must hold before an action, so the goal holds after it?
def move(b, x, y):
    return {"name": "Move(" + b + ", " + x + ", " + y + ")",
            "pre": {"On(" + b + ", " + x + ")", "Clear(" + b + ")", "Clear(" + y + ")"},
            "add": {"On(" + b + ", " + y + ")"} | ({"Clear(" + x + ")"} if x != "Table" else set()),
            "del": {"On(" + b + ", " + x + ")", "Clear(" + y + ")"}}

def regress(goal, a):
    if not (a["add"] & goal):
        return None                     # not relevant: it achieves no part of the goal
    if a["del"] & goal:
        return None                     # not consistent: it would undo part of the goal
    return (goal - a["add"]) | a["pre"]

goal = {"On(A, B)", "On(B, C)"}
for a in [move("A", "Table", "B"), move("B", "Table", "C"), move("C", "A", "B")]:
    g = regress(goal, a)
    print(a["name"], "->", sorted(g) if g else "not usable")

## Your turn

### Task 1: Apply a STRIPS action

A state is a Python set of fact strings. An action is a dictionary with sets "pre", "add" and "del". Write `apply(state, action)`: return None if some precondition is missing, else the new state (delete first, then add).

Hint: Check every precondition. Then copy the state, remove the delete list and add the add list.

In [ ]:
def apply(state, action):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("sorted(apply({'On(C, A)', 'On(A, Table)', 'On(B, Table)', 'Clear(C)', 'Clear(B)'}, {'pre': {'On(C, A)', 'Clear(C)'}, 'add': {'On(C, Table)', 'Clear(A)'}, 'del': {'On(C, A)'}}))", ['Clear(A)', 'Clear(B)', 'Clear(C)', 'On(A, Table)', 'On(B, Table)', 'On(C, Table)']), ("apply({'On(A, Table)'}, {'pre': {'Clear(A)'}, 'add': set(), 'del': set()})", None), ("sorted(apply({'P', 'Q'}, {'pre': {'P'}, 'add': {'R'}, 'del': {'P'}}))", ['Q', 'R'])])

### Task 2: Regress a goal through an action

Write `regress(goal, action)` with sets as in the last task. Return None if the action is not relevant (adds nothing in the goal) or not consistent (deletes something in the goal). Otherwise return (goal − add) ∪ pre.

Hint: Use set intersection (&) to test relevance and consistency, then set difference and union.

In [ ]:
def regress(goal, action):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("sorted(regress({'On(A, B)', 'On(B, C)'}, {'pre': {'On(A, Table)', 'Clear(A)', 'Clear(B)'}, 'add': {'On(A, B)'}, 'del': {'On(A, Table)', 'Clear(B)'}}))", ['Clear(A)', 'Clear(B)', 'On(A, Table)', 'On(B, C)']), ("regress({'On(A, B)'}, {'pre': {'X'}, 'add': {'Y'}, 'del': set()})", None), ("regress({'P', 'Q'}, {'pre': {'R'}, 'add': {'P'}, 'del': {'Q'}})", None), ("sorted(regress({'P'}, {'pre': {'R', 'S'}, 'add': {'P'}, 'del': set()}))", ['R', 'S'])])

### Task 3: A blocks world planner

A state is a dictionary from block to what it sits on ("Table" or another block). Write `plan_blocks(start, goal)` that returns a shortest list of moves as (block, to) pairs, using breadth first search. A move picks a clear block and puts it on the table (if it is not there already) or on another clear block. goal is a dictionary of required On facts.

Hint: A block is clear if no block sits on it. Keep a queue of (state, plan) and a list of seen states.

In [ ]:
def plan_blocks(start, goal):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("len(plan_blocks({'A': 'Table', 'B': 'Table', 'C': 'A'}, {'A': 'B', 'B': 'C'}))", 3), ("plan_blocks({'A': 'Table', 'B': 'Table'}, {'A': 'B'})", [('A', 'B')]), ("plan_blocks({'A': 'B', 'B': 'Table'}, {'A': 'B'})", []), ("len(plan_blocks({'A': 'B', 'B': 'C', 'C': 'Table'}, {'C': 'B', 'B': 'A'}))", 3)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Apply a STRIPS action

Test 1 is MoveToTable(C, A) in the Sussman start.

In [ ]:
def apply(state, action):
    for p in action["pre"]:
        if p not in state:
            return None
    new = set(state)
    for d in action["del"]:
        new.discard(d)
    for a in action["add"]:
        new.add(a)
    return new

In [ ]:
check([("sorted(apply({'On(C, A)', 'On(A, Table)', 'On(B, Table)', 'Clear(C)', 'Clear(B)'}, {'pre': {'On(C, A)', 'Clear(C)'}, 'add': {'On(C, Table)', 'Clear(A)'}, 'del': {'On(C, A)'}}))", ['Clear(A)', 'Clear(B)', 'Clear(C)', 'On(A, Table)', 'On(B, Table)', 'On(C, Table)']), ("apply({'On(A, Table)'}, {'pre': {'Clear(A)'}, 'add': set(), 'del': set()})", None), ("sorted(apply({'P', 'Q'}, {'pre': {'P'}, 'add': {'R'}, 'del': {'P'}}))", ['Q', 'R'])])

### Solution to task 2: Regress a goal through an action

Test 3: the action adds P but deletes Q, which the goal also needs, so it cannot be the last step.

In [ ]:
def regress(goal, action):
    if len(goal & action["add"]) == 0:
        return None
    if len(goal & action["del"]) > 0:
        return None
    return (goal - action["add"]) | action["pre"]

In [ ]:
check([("sorted(regress({'On(A, B)', 'On(B, C)'}, {'pre': {'On(A, Table)', 'Clear(A)', 'Clear(B)'}, 'add': {'On(A, B)'}, 'del': {'On(A, Table)', 'Clear(B)'}}))", ['Clear(A)', 'Clear(B)', 'On(A, Table)', 'On(B, C)']), ("regress({'On(A, B)'}, {'pre': {'X'}, 'add': {'Y'}, 'del': set()})", None), ("regress({'P', 'Q'}, {'pre': {'R'}, 'add': {'P'}, 'del': {'Q'}})", None), ("sorted(regress({'P'}, {'pre': {'R', 'S'}, 'add': {'P'}, 'del': set()}))", ['R', 'S'])])

### Solution to task 3: A blocks world planner

Test 4 turns a tower upside down. At the start A is on top and C is at the bottom. The goal puts C on B and B on A, so A must end at the bottom. The shortest plan has three moves: A to the table, B onto A, then C onto B.

In [ ]:
def plan_blocks(start, goal):
    queue = [(start, [])]
    seen = [start]
    while queue:
        state, plan = queue.pop(0)
        done = True
        for b in goal:
            if state[b] != goal[b]:
                done = False
        if done:
            return plan
        below = set(state.values())
        free = []
        for b in sorted(state):
            if b not in below:
                free.append(b)
        for b in free:
            targets = []
            if state[b] != "Table":
                targets.append("Table")
            for y in free:
                if y != b:
                    targets.append(y)
            for to in targets:
                new = dict(state)
                new[b] = to
                if new not in seen:
                    seen.append(new)
                    queue.append((new, plan + [(b, to)]))
    return None

In [ ]:
check([("len(plan_blocks({'A': 'Table', 'B': 'Table', 'C': 'A'}, {'A': 'B', 'B': 'C'}))", 3), ("plan_blocks({'A': 'Table', 'B': 'Table'}, {'A': 'B'})", [('A', 'B')]), ("plan_blocks({'A': 'B', 'B': 'Table'}, {'A': 'B'})", []), ("len(plan_blocks({'A': 'B', 'B': 'C', 'C': 'Table'}, {'C': 'B', 'B': 'A'}))", 3)])